The previous notebook covered the foundations of object-oriented programming in Python. This one goes deeper — into the machinery that professional Python libraries are built with. Abstract base classes enforce interface contracts. Protocols bring structural subtyping. `__slots__` cuts memory usage. Descriptors are the mechanism behind properties and classmethods. Context managers let objects manage resources with `with`. Metaclasses control class creation itself. These tools rarely appear in application code written from scratch, but you will encounter them frequently when reading framework code, and occasionally need them when building reusable libraries.

## Abstract Base Classes

An **abstract base class (ABC)** defines an interface — a set of methods that every concrete subclass must implement. Trying to instantiate a class that inherits from an ABC without implementing all abstract methods raises a `TypeError`.

ABCs formalise the contract between a base class and its subclasses. They are the Python equivalent of interfaces or abstract classes in Java/C++. Use them when you want to define a common API that multiple implementations must follow.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class for all 2D shapes."""

    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""

    @abstractmethod
    def perimeter(self) -> float:
        """Return the perimeter of the shape."""

    def describe(self):
        # Concrete method — available to all subclasses
        return (f"{type(self).__name__}: "
                f"area={self.area():.2f}, perimeter={self.perimeter():.2f}")


# Trying to instantiate the ABC directly raises TypeError
try:
    s = Shape()
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
import math

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width  = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


# A subclass that forgets to implement an abstract method
class Blob(Shape):
    pass


shapes = [Circle(5), Rectangle(4, 6)]
for shape in shapes:
    print(shape.describe())

# isinstance works with ABCs
print(isinstance(Circle(1), Shape))  # True

try:
    Blob()   # TypeError — area() and perimeter() not implemented
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# Abstract properties
class Animal(ABC):
    @property
    @abstractmethod
    def sound(self) -> str:
        """The sound this animal makes."""

    def speak(self):
        return f"{type(self).__name__} says {self.sound!r}"


class Dog(Animal):
    @property
    def sound(self):
        return "Woof"


class Cat(Animal):
    @property
    def sound(self):
        return "Meow"


for animal in [Dog(), Cat()]:
    print(animal.speak())

## Protocols — Structural Subtyping

ABCs enforce a contract through explicit inheritance. **Protocols** (PEP 544, Python 3.8+) take a different approach: **structural subtyping** — an object satisfies a Protocol simply by having the right attributes and methods, without inheriting from anything.

This is what Python's dynamic nature has always allowed informally (duck typing). Protocols make it explicit and statically checkable by type checkers like mypy. The standard library's `Iterable`, `Sized`, `Callable` and dozens of others are Protocols.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str:
        ...

    def bounding_box(self) -> tuple[float, float, float, float]:
        ...


# These classes do NOT inherit from Drawable — they just have the right methods
class Sprite:
    def __init__(self, x, y, w, h, label):
        self.x, self.y, self.w, self.h = x, y, w, h
        self.label = label

    def draw(self):
        return f"<Sprite {self.label!r} at ({self.x},{self.y})>"

    def bounding_box(self):
        return (self.x, self.y, self.x + self.w, self.y + self.h)


class TextLabel:
    def __init__(self, x, y, text):
        self.x, self.y, self.text = x, y, text

    def draw(self):
        return f"<Text {self.text!r} at ({self.x},{self.y})>"

    def bounding_box(self):
        return (self.x, self.y, self.x + len(self.text) * 8, self.y + 16)


def render_all(items: list[Drawable]) -> None:
    for item in items:
        print(item.draw(), "bbox:", item.bounding_box())


scene = [Sprite(10, 20, 64, 64, "player"), TextLabel(0, 0, "Score: 42")]
render_all(scene)

# runtime_checkable allows isinstance checks
print(isinstance(Sprite(0,0,1,1,"x"), Drawable))  # True

## `__slots__`

By default every instance stores its attributes in a `__dict__` (a dictionary). This is flexible but uses significant memory. Declaring `__slots__` replaces the per-instance dict with a fixed set of slots — more like C struct fields.

Benefits:
- **~30–50% lower memory** per instance (no dict overhead).
- **Slightly faster** attribute access.
- **Prevents typos** — assigning an undeclared attribute raises `AttributeError`.

Cost: you cannot add arbitrary attributes at runtime, and slots interact with inheritance in ways you need to understand.

In [ ]:
import sys

class PointDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


pd = PointDict(1.0, 2.0)
ps = PointSlots(1.0, 2.0)

print(f"With __dict__:  {sys.getsizeof(pd) + sys.getsizeof(pd.__dict__)} bytes")
print(f"With __slots__: {sys.getsizeof(ps)} bytes")

# __dict__ class has it; slots class does not
print(hasattr(pd, "__dict__"))   # True
print(hasattr(ps, "__dict__"))   # False

# Attempting to set an undeclared attribute on a slots class raises AttributeError
try:
    ps.z = 3.0
except AttributeError as e:
    print(f"AttributeError: {e}")

In [ ]:
# Memory impact at scale
N = 100_000

dict_points  = [PointDict(i, i)  for i in range(N)]
slots_points = [PointSlots(i, i) for i in range(N)]

dict_mem  = sum(sys.getsizeof(p) + sys.getsizeof(p.__dict__) for p in dict_points)
slots_mem = sum(sys.getsizeof(p) for p in slots_points)

print(f"100k PointDict  objects: {dict_mem / 1024 / 1024:.1f} MB")
print(f"100k PointSlots objects: {slots_mem / 1024 / 1024:.1f} MB")
print(f"Memory saved: {(dict_mem - slots_mem) / 1024 / 1024:.1f} MB")

## Descriptors

A **descriptor** is any object that defines `__get__`, `__set__`, or `__delete__`. When a descriptor is assigned as a class attribute, Python calls those methods whenever the attribute is accessed, assigned, or deleted on an instance.

This is the underlying mechanism that makes `@property`, `@classmethod`, `@staticmethod`, and `__slots__` work. Understanding descriptors means understanding how Python's attribute lookup actually functions.

- **Data descriptor**: defines both `__get__` and `__set__` (or `__delete__`). Takes priority over instance `__dict__`.
- **Non-data descriptor**: defines only `__get__`. Instance `__dict__` takes priority.

In [ ]:
class Validated:
    """Descriptor that validates a numeric value is within [min_val, max_val]."""

    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val
        self.attr_name = None   # set by __set_name__

    def __set_name__(self, owner, name):
        # Called when the class is defined — gives the descriptor its attribute name
        self.attr_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self   # accessed on the class, not an instance
        return getattr(instance, self.attr_name, None)

    def __set__(self, instance, value):
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(
                f"{self.attr_name[1:]!r} must be in [{self.min_val}, {self.max_val}], got {value}"
            )
        setattr(instance, self.attr_name, value)


class Sensor:
    temperature = Validated(-40, 85)   # descriptor as class attribute
    humidity    = Validated(0, 100)

    def __init__(self, temperature, humidity):
        self.temperature = temperature   # calls Validated.__set__
        self.humidity    = humidity

    def __repr__(self):
        return f"Sensor(temp={self.temperature}, humidity={self.humidity})"


s = Sensor(22, 55)
print(s)

s.temperature = 30
print(s)

try:
    s.temperature = 100   # exceeds max of 85
except ValueError as e:
    print(f"ValueError: {e}")

## Context Managers — `__enter__` and `__exit__`

The `with` statement calls `__enter__` on entry and `__exit__` on exit — even if an exception occurs. Implementing these two methods turns any class into a context manager.

`__exit__` receives three arguments after `self`: the exception type, exception value, and traceback (all `None` if no exception occurred). Returning a truthy value suppresses the exception; returning `None` or `False` lets it propagate.

In [ ]:
import time

class Timer:
    """Context manager that measures elapsed time."""

    def __enter__(self):
        self._start = time.perf_counter()
        return self   # the value bound by 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        # Return False / None to let exceptions propagate
        return False


with Timer() as t:
    total = sum(i ** 2 for i in range(1_000_000))

print(f"Result: {total}")
print(f"Elapsed: {t.elapsed:.4f}s")

In [ ]:
class ManagedConnection:
    """Simulates a database connection that must be closed after use."""

    def __init__(self, host):
        self.host = host

    def __enter__(self):
        print(f"[+] Connected to {self.host}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"[-] Exception occurred: {exc_val}")
        print(f"[-] Disconnected from {self.host}")
        return False   # don't suppress exceptions

    def query(self, sql):
        print(f"    Executing: {sql!r}")
        return [{"id": 1}, {"id": 2}]


# Normal exit
with ManagedConnection("db.example.com") as conn:
    rows = conn.query("SELECT * FROM users")
    print(f"    Got {len(rows)} rows")

print()

# Exception still triggers __exit__
try:
    with ManagedConnection("db.example.com") as conn:
        conn.query("SELECT 1")
        raise RuntimeError("something went wrong")
except RuntimeError:
    print("Exception propagated and was caught outside")

In [ ]:
# contextlib.contextmanager — write a context manager as a generator function
from contextlib import contextmanager

@contextmanager
def temporary_attribute(obj, attr, value):
    """Temporarily override an attribute, restoring the original on exit."""
    original = getattr(obj, attr, None)
    setattr(obj, attr, value)
    try:
        yield obj
    finally:
        setattr(obj, attr, original)


class Config:
    debug = False

cfg = Config()
print(f"Before: debug={cfg.debug}")

with temporary_attribute(cfg, "debug", True):
    print(f"Inside: debug={cfg.debug}")

print(f"After:  debug={cfg.debug}")

## `__new__` — Controlling Instance Creation

`__init__` initialises an already-created instance. `__new__` is called before `__init__` and is responsible for **creating** the instance. Overriding `__new__` lets you control exactly what object gets created — useful for singletons, cached instances, and immutable subclasses.

`__new__` is a static method that receives the class as its first argument and must return an instance. If it returns an instance of the class, `__init__` is called on it automatically.

In [ ]:
# Singleton — only one instance ever exists
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value):
        self.value = value


a = Singleton(1)
b = Singleton(2)

print(a is b)        # True — same object
print(a.value)       # 2 — __init__ ran again on the existing instance
print(id(a) == id(b))  # True

In [ ]:
# __new__ for subclassing immutable types like int or str
class PositiveInt(int):
    """An integer that is guaranteed to be positive."""

    def __new__(cls, value):
        if value <= 0:
            raise ValueError(f"PositiveInt requires a positive value, got {value}")
        return super().__new__(cls, value)


n = PositiveInt(5)
print(n, type(n))        # 5 <class '__main__.PositiveInt'>
print(n + 3)             # 8 — arithmetic works (int operations)

try:
    PositiveInt(-1)
except ValueError as e:
    print(e)

## `__init_subclass__` — Hooks for Subclassing

`__init_subclass__` is called on the **parent** class whenever a new subclass is defined. It lets the parent register, validate, or modify the subclass without needing a metaclass. This is a clean way to enforce rules on subclasses or build plugin/registry patterns.

In [ ]:
class Plugin:
    """Base class that automatically registers all subclasses."""
    _registry: dict[str, type] = {}

    def __init_subclass__(cls, name: str = "", **kwargs):
        super().__init_subclass__(**kwargs)
        plugin_name = name or cls.__name__.lower()
        Plugin._registry[plugin_name] = cls
        print(f"Registered plugin: {plugin_name!r}")

    @classmethod
    def get(cls, name: str) -> "Plugin":
        plugin_cls = cls._registry.get(name)
        if plugin_cls is None:
            raise KeyError(f"No plugin named {name!r}")
        return plugin_cls()


class JSONPlugin(Plugin, name="json"):
    def run(self): return "Serialising to JSON"

class CSVPlugin(Plugin, name="csv"):
    def run(self): return "Serialising to CSV"

class YAMLPlugin(Plugin):   # name defaults to class name lowercased
    def run(self): return "Serialising to YAML"


print()
print("Registry:", list(Plugin._registry.keys()))
print(Plugin.get("json").run())
print(Plugin.get("yamlplugin").run())

## Metaclasses

A **metaclass** is a class whose instances are classes. Just as a regular class controls what instances of it look like, a metaclass controls what classes look like.

When Python executes a `class` statement, it calls a metaclass to build the class object. The default metaclass is `type`. You can override it with `metaclass=` in the class definition.

Metaclasses are powerful but complex. Before reaching for one, consider whether `__init_subclass__`, a class decorator, or an ABC achieves the same goal with less machinery.

In [ ]:
# type is the default metaclass — you can create classes dynamically with it
# type(name, bases, namespace)
Dog = type("Dog", (object,), {
    "species": "Canis lupus familiaris",
    "speak": lambda self: "Woof!",
})

rex = Dog()
print(rex.species)
print(rex.speak())
print(type(Dog))   # <class 'type'>

In [ ]:
# Custom metaclass — enforce that all public methods have docstrings
class DocstringEnforcer(type):
    def __new__(mcs, name, bases, namespace):
        for attr_name, attr_value in namespace.items():
            if callable(attr_value) and not attr_name.startswith("_"):
                if not attr_value.__doc__:
                    raise TypeError(
                        f"{name}.{attr_name}() is missing a docstring"
                    )
        return super().__new__(mcs, name, bases, namespace)


# This class is fine — all public methods have docstrings
class WellDocumented(metaclass=DocstringEnforcer):
    def process(self, data):
        """Process the given data and return a result."""
        return data

print("WellDocumented defined OK")

# This class fails at definition time — not at instantiation
try:
    class PoorlyDocumented(metaclass=DocstringEnforcer):
        def process(self, data):   # no docstring!
            return data
except TypeError as e:
    print(f"TypeError: {e}")

## Class Decorators

A class decorator is a callable applied to a class definition — it receives the class object and returns a (possibly modified) class. Class decorators are simpler than metaclasses for many use cases and more explicit about what they change. `@dataclass` is the most widely used class decorator in the standard library.

In [ ]:
# A class decorator that adds a .to_dict() method to any class
def add_to_dict(cls):
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith("_")}
    cls.to_dict = to_dict
    return cls


@add_to_dict
class Order:
    def __init__(self, order_id, product, qty):
        self.order_id = order_id
        self.product  = product
        self.qty      = qty


@add_to_dict
class Customer:
    def __init__(self, name, email):
        self.name  = name
        self.email = email


order    = Order("ORD-001", "Widget", 3)
customer = Customer("Alice", "alice@example.com")

print(order.to_dict())
print(customer.to_dict())

In [ ]:
# Decorator that validates all __init__ arguments against type annotations
def enforce_types(cls):
    original_init = cls.__init__
    annotations   = cls.__annotations__

    def __init__(self, *args, **kwargs):
        # bind args to parameter names using inspect
        import inspect
        sig    = inspect.signature(original_init)
        bound  = sig.bind(self, *args, **kwargs)
        bound.apply_defaults()
        for param, value in list(bound.arguments.items())[1:]:   # skip 'self'
            if param in annotations and not isinstance(value, annotations[param]):
                raise TypeError(
                    f"{param!r} must be {annotations[param].__name__}, "
                    f"got {type(value).__name__}"
                )
        original_init(self, *args, **kwargs)

    cls.__init__ = __init__
    return cls


@enforce_types
class Rectangle:
    width:  float
    height: float

    def __init__(self, width: float, height: float):
        self.width  = width
        self.height = height


r = Rectangle(4.0, 3.0)
print(r.width, r.height)

try:
    Rectangle("wide", 3.0)   # type mismatch
except TypeError as e:
    print(f"TypeError: {e}")

## Summary

- **Abstract Base Classes** (`abc.ABC`, `@abstractmethod`): define interfaces that subclasses must implement. Attempting to instantiate an incomplete subclass raises `TypeError` at construction time. Abstract properties combine `@property` and `@abstractmethod`.
- **Protocols** (`typing.Protocol`): structural subtyping — an object satisfies the protocol simply by having the right attributes and methods, without explicit inheritance. `@runtime_checkable` enables `isinstance` checks.
- **`__slots__`**: replaces the per-instance `__dict__` with fixed memory slots. Cuts memory by 30–50% and slightly speeds attribute access. Prevents arbitrary attribute assignment.
- **Descriptors** (`__get__`, `__set__`, `__set_name__`): the mechanism behind `@property`, `@classmethod`, and `@staticmethod`. A descriptor class assigned as a class attribute intercepts all attribute access, assignment, and deletion on instances.
- **Context managers** (`__enter__`, `__exit__`): `__enter__` runs on `with` entry and returns the value bound by `as`. `__exit__` runs on exit — exception or not. Return `True` from `__exit__` to suppress an exception. `@contextmanager` lets you write one as a generator function.
- **`__new__`**: called before `__init__`, responsible for creating the instance. Use for singletons, cached instances, and subclassing immutable types (`int`, `str`, `tuple`).
- **`__init_subclass__`**: called on the parent when a subclass is defined. Use for plugin registries, subclass validation, and framework hooks — simpler than a metaclass for most use cases.
- **Metaclasses**: a class whose instances are classes. The default is `type`. Override `__new__` or `__init__` on a metaclass to control class creation. Prefer `__init_subclass__` or class decorators when they suffice.
- **Class decorators**: a callable applied to a class object. Simpler and more explicit than metaclasses for adding methods, enforcing conventions, or wrapping `__init__`.